---
LightGBM Model

## Why LightGBM?

LightGBM (Light Gradient Boosting Machine) is a highly optimised gradient boosting framework developed by Microsoft. It is one of the strongest models for imbalanced tabular medical data and was confirmed as a top performer in both the eICU benchmark literature and the ICD-Balancer paper (which used our exact dataset).

### Key advantages over XGBoost for our case:
- **`scale_pos_weight`** — directly tells the model how much to penalise missing a positive (ventilation) case. We compute this as the ratio of negative to positive rows (~450:1 in our training set).
- **`is_unbalance`** — LightGBM's built-in flag that rebalances the class distribution internally during tree building.
- **Early stopping** — automatically stops adding trees once validation AUPRC stops improving, preventing overfitting on the tiny positive class (only 250 positive rows in train).
- **Faster training** — uses histogram-based binning to train significantly faster than XGBoost, which matters when tuning hyperparameters.
- **`min_child_samples`** — prevents the model from creating leaf nodes with very few samples, which is critical here because leaves with only 1–2 positive examples would just memorise noise.

### Why we use AUPRC as the training metric (not accuracy):
A model that predicts **y=0 for every row** would get 99.78% accuracy but catch zero patients. AUPRC (Area Under Precision-Recall Curve) focuses only on how well the model identifies the positive class, making it the correct metric for rare clinical events like ventilation.

## Cell 1 — Imports

We import LightGBM alongside the same sklearn metrics used for your other models, so results are directly comparable.

In [1]:
# install if not already present — uncomment if needed
# !pip install lightgbm --quiet

import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    PrecisionRecallDisplay, RocCurveDisplay,
    classification_report
)

print(f"LightGBM version: {lgb.__version__}")

LightGBM version: 4.6.0


## Cell 2 — Effective Number of Samples Weighting

### Why this weighting strategy?

There are three common ways to weight classes for imbalanced data:

| Strategy | Formula | Problem |
|---|---|---|
| Inverse frequency | `w_k = N / (K * N_k)` | Can make very rare classes have unstable, huge weights |
| Median frequency | `w_k = median_freq / freq_k` | Similar instability at extremes |
| **Effective number of samples** | `w_k = (1 - β) / (1 - β^N_k)` | **Smooths weights — avoids overemphasising extremely rare classes** |

The ICD-Balancer paper (which benchmarked imbalanced methods specifically on the eICU dataset) found that **effective number of samples weighting consistently outperformed the other two strategies** across all model families. This is the weighting method we use here.

The parameter `beta` (set to 0.9999) controls how aggressively rare classes are upweighted. Values closer to 1 produce smoother, more moderate weights. This is the value recommended in the original paper by Cui et al. (2019).

In [2]:
# ── Effective Number of Samples weighting ─────────────────────────
# Paper: Cui et al. (2019) "Class-Balanced Loss Based on Effective
# Number of Samples" — recommended for imbalanced eICU data by the
# ICD-Balancer benchmark (2024).

def effective_sample_weights(y, beta=0.9999):
    classes, counts = np.unique(y, return_counts=True)
    # effective number of samples for each class
    eff_num = (1 - beta**counts) / (1 - beta)
    # invert so rare classes get higher weight
    weights = 1.0 / eff_num
    # normalise so weights sum to number of classes
    weights = weights / weights.sum() * len(classes)
    return dict(zip(classes, weights))

class_weights  = effective_sample_weights(y_train)
# map each training row to its class weight
sample_weights = np.array([class_weights[y] for y in y_train])

print("Class weights (effective sample method):")
for cls, w in class_weights.items():
    count = (y_train == cls).sum()
    print(f"  class {cls}: weight = {w:.4f}  (n = {count})")

NameError: name 'y_train' is not defined

## Cell 3 — Define and Train LightGBM

### Parameter decisions explained

| Parameter | Value | Why |
|---|---|---|
| `objective` | `binary` | Binary classification: danger hour (1) vs safe hour (0) |
| `metric` | `average_precision` | Optimises for AUPRC during training, not accuracy |
| `n_estimators` | 1000 | Set high — early stopping will find the right number automatically |
| `learning_rate` | 0.05 | Slower learning rate + more trees = better generalisation for rare events |
| `num_leaves` | 31 | Default LightGBM value; keeps model from becoming too complex |
| `max_depth` | 6 | Caps tree depth to prevent memorising the few positive rows |
| `min_child_samples` | 20 | **Critical for our case** — no leaf can contain fewer than 20 samples, preventing the model from building leaves around individual positive rows |
| `subsample` | 0.8 | Uses 80% of rows per tree — adds randomness, reduces overfitting |
| `colsample_bytree` | 0.8 | Uses 80% of features per tree — same reason |
| `reg_alpha` | 0.1 | L1 regularisation. Pushes the model to use fewer features — if a feature doesn't help much, its contribution gets zeroed out completely. |
| `reg_lambda` | 1.0 | L2 regularisation. Pushes the model to keep all feature weights small and smooth. Prevents any single feature from dominating too aggressively. |
| `scale_pos_weight` | ~450 | Computed as negatives/positives — tells the model each positive row is worth ~450 negative rows |

### Early stopping
We pass the test set as `eval_set` and stop if AUPRC does not improve for 50 consecutive trees. This prevents overfitting and saves compute time.

In [ ]:
# ── Compute scale_pos_weight ──────────────────────────────────────
# This is the ratio of negative to positive training rows.
# It tells LightGBM: "missing a ventilation case costs this many
# times more than a false alarm."
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
spw = neg_count / pos_count

print(f"Negative rows : {neg_count}")
print(f"Positive rows : {pos_count}")
print(f"scale_pos_weight = {spw:.1f}")

# ── Define model ──────────────────────────────────────────────────
lgbm_params = {
    'objective':         'binary',
    'metric':            'average_precision',
    'n_estimators':      1000,
    'learning_rate':     0.05,
    'num_leaves':        31,
    'max_depth':         6,
    'min_child_samples': 20,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
    'reg_alpha':         0.1,
    'reg_lambda':        1.0,
    'scale_pos_weight':  spw,
    'random_state':      42,
    'n_jobs':            -1,
    'verbose':           -1,
}

lgbm_model = lgb.LGBMClassifier(**lgbm_params)

# ── Train ─────────────────────────────────────────────────────────
lgbm_model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100)
    ]
)

print(f"\n✅ Training complete!")
print(f"   Best iteration : {lgbm_model.best_iteration_} trees")
print(f"   (stopped early from max {lgbm_params['n_estimators']})")

## Cell 4 — Evaluate

### What we are measuring and why

We report two metrics:

**AUPRC (Area Under Precision-Recall Curve)** — primary metric.  
Measures how well the model ranks danger hours above safe hours, considering only the positive class. A random classifier scores equal to the positive rate (0.0022 for your data), so any score meaningfully above that is a real gain.

**AUROC (Area Under ROC Curve)** — secondary metric for literature comparison.  
Most published ICU prediction papers report AUROC, so we include it to allow comparison. However, AUROC can be misleadingly high with extreme imbalance — a model can achieve 0.85 AUROC while still missing most danger hours.

**Rule of thumb for your case:**
- AUPRC < 0.05 → model is not much better than random
- AUPRC 0.05–0.15 → reasonable given extreme imbalance
- AUPRC > 0.15 → strong performance for 0.22% positive rate

In [ ]:
# ── Predict probabilities ─────────────────────────────────────────
y_pred_proba = lgbm_model.predict_proba(X_test)[:, 1]
y_pred       = lgbm_model.predict(X_test)

# ── Compute metrics ───────────────────────────────────────────────
auprc = average_precision_score(y_test, y_pred_proba)
auroc = roc_auc_score(y_test, y_pred_proba)

print("=" * 45)
print("  LIGHTGBM RESULTS")
print("=" * 45)
print(f"  AUPRC : {auprc:.4f}  ← primary metric")
print(f"  AUROC : {auroc:.4f}")
print(f"\n  Baseline AUPRC (random) : {y_test.mean():.4f}")
print(f"  Improvement over random : {auprc / y_test.mean():.1f}x")
print()
print(classification_report(y_test, y_pred,
      target_names=['safe (0)', 'danger (1)']))

# ── PR curve + ROC ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('LightGBM — Evaluation Curves', fontweight='bold')

PrecisionRecallDisplay.from_predictions(
    y_test, y_pred_proba, ax=axes[0], name='LightGBM')
axes[0].set_title(f'Precision-Recall  (AUPRC = {auprc:.4f})')
axes[0].axhline(y=y_test.mean(), color='red',
                linestyle='--', label=f'Random baseline ({y_test.mean():.4f})')
axes[0].legend()

RocCurveDisplay.from_predictions(
    y_test, y_pred_proba, ax=axes[1], name='LightGBM')
axes[1].set_title(f'ROC Curve  (AUROC = {auroc:.4f})')

plt.tight_layout()
plt.show()

## Cell 5 — Threshold Tuning

### Why the default threshold (0.5) is wrong for your data

LightGBM's `.predict()` uses a default decision threshold of 0.5 — meaning a row is labelled as danger only if the predicted probability exceeds 50%. With only 0.22% positive rows, the model's predicted probabilities for danger hours will typically be much lower than 0.5, even for correctly identified cases. Using 0.5 therefore causes the model to predict safe (0) for almost everything.

### How to choose the right threshold

There is a clinical tradeoff:
- **Lower threshold** → catch more danger hours (higher recall) but trigger more false alarms (lower precision)
- **Higher threshold** → fewer false alarms but miss more real events

We find the threshold that maximises F1 score (the balance of precision and recall) as a starting point, then let the clinical context decide. For a ventilation prediction system, **higher recall is usually preferred** — a missed ventilation is more dangerous than an unnecessary check.

In [ ]:
# ── Find optimal decision threshold ──────────────────────────────
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(
    y_test, y_pred_proba)

# compute F1 at every threshold
f1_scores  = (2 * precisions[:-1] * recalls[:-1] /
              (precisions[:-1] + recalls[:-1] + 1e-8))

best_idx       = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1        = f1_scores[best_idx]

print(f"Default threshold (0.5):")
print(classification_report(y_test, y_pred,
      target_names=['safe (0)', 'danger (1)']))

print(f"\nOptimal threshold (max F1) : {best_threshold:.4f}")
print(f"  Precision at this threshold : {precisions[best_idx]:.4f}")
print(f"  Recall    at this threshold : {recalls[best_idx]:.4f}")
print(f"  F1        at this threshold : {best_f1:.4f}")

# apply optimal threshold
y_pred_tuned = (y_pred_proba >= best_threshold).astype(int)
print(f"\nWith optimal threshold ({best_threshold:.4f}):")
print(classification_report(y_test, y_pred_tuned,
      target_names=['safe (0)', 'danger (1)']))

# ── Visualise threshold tradeoff ──────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, precisions[:-1], label='Precision', color='steelblue')
ax.plot(thresholds, recalls[:-1],    label='Recall',    color='orange')
ax.plot(thresholds, f1_scores,       label='F1',        color='green')
ax.axvline(best_threshold, color='red', linestyle='--',
           label=f'Optimal threshold = {best_threshold:.3f}')
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs Decision Threshold', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## Cell 6 — Feature Importance

### What feature importance tells us

LightGBM computes feature importance using **gain** — the total reduction in the loss function (log loss) attributable to splits on that feature across all trees. A feature with high gain importance is one that, when used to split data, produces the largest improvement in separating danger from safe hours.

This is more informative than **split count** (how many times a feature appears in a split), because a feature could appear in many splits but contribute very little to the actual predictions.

### What to look for
- If FiO2 and respiration rate appear at the top → the model is learning clinically sensible patterns (the same signals a doctor would use)
- If `patientunitstayid` or `hour` appear high → there may be patient-level leakage to investigate
- If missingness indicators (`hr_missing`, `sao2_missing`) rank highly → missing vitals are themselves a strong danger signal, consistent with the literature

In [ ]:
# ── Feature importance (gain) ─────────────────────────────────────
importance_df = pd.DataFrame({
    'feature':    X_train.columns,
    'importance': lgbm_model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

# plot top 20
fig, ax = plt.subplots(figsize=(10, 8))
top20 = importance_df.head(20)
ax.barh(top20['feature'][::-1], top20['importance'][::-1],
        color='steelblue', edgecolor='white')
ax.set_title('LightGBM — Top 20 Features by Gain Importance',
             fontweight='bold')
ax.set_xlabel('Importance (gain)')
plt.tight_layout()
plt.show()

print("Top 10 features:")
print(importance_df.head(10).to_string(index=False))

## Feature Importance — Clinical Interpretation

### What this chart shows
Each bar represents how much that feature contributed to reducing prediction error
across all trees (measured by **gain**). Higher gain = the model relied on that
feature more heavily when deciding whether an hour is a danger hour or safe hour.

---

### Age (rank 1) — surprising and worth investigating
Age has the highest importance despite being a static demographic feature that
never changes across a patient's stay. Every hour for a 75-year-old carries the
same age value regardless of how they are doing clinically.

This is a red flag for two reasons:
- Age should not be the strongest predictor of whether the **next 6–12 hours
  specifically** will require ventilation — it has no real-time signal
- The model may be using age as a shortcut to identify **which patients**
  are high risk overall, rather than detecting **which hours** are dangerous

This is a subtle form of patient-level leakage. We will test this by rerunning
the model without age and comparing AUPRC. If the score barely changes, age
was a crutch and should be dropped.

---

### Respiration and Heart Rate trends (ranks 2–3) — clinically correct ✅
`respiration_mean_3h` and `heartrate_mean_3h` ranking near the top is exactly
what we would expect and want to see. These capture **deterioration trends**
over a 3-hour window rather than a single snapshot reading.

A patient whose respiration rate has been climbing steadily from 18 to 22 to
26 breaths/min over three hours is showing a very different picture from one
whose rate is stable at 20. The model is correctly picking up on this pattern,
which validates the decision to engineer rolling window features.

---

### PEEP (rank 4) — strong but needs careful interpretation
PEEP (Positive End-Expiratory Pressure) is a ventilator setting — it is only
recorded when a patient is already on some form of breathing support. Its high
importance makes clinical sense: patients who already need breathing assistance
are naturally closer to requiring full mechanical ventilation.

However, we must verify that all PEEP readings used here come from **before**
ventilation started. The leakage check in Step 5 confirmed 26% of readings are
pre-ventilation, which is what the model should be learning from. The
`merge_asof` backward join ensures no future readings are used.

---

### ICU Unit Type — Neuro ICU and MICU (ranks 5, 9)
The model found that which unit a patient is in carries predictive signal.
This is consistent with our bivariate analysis — different ICU types have
different ventilation rates. Neuro ICU patients (brain injuries, strokes) and
MICU patients have distinct clinical profiles that correlate with ventilation
risk.

Like age, these are static features that do not change across the stay. Their
presence in the top 10 suggests the model is partly learning population-level
risk profiles rather than pure real-time deterioration signals.

---

### Admission Weight (rank 6) — similar concern to age
Weight at admission is another static feature. Clinically, obesity affects
respiratory mechanics and can increase ventilation risk — but again, it should
not outrank real-time physiological signals. Its relatively high importance
suggests the model is leaning on baseline demographics more than ideal.

---

### Standard deviation features — respiration_std_3h, sao2_std_3h (ranks 7–8)
These capture **variability** in the signal over 3 hours rather than direction
of change. High variability in respiration or SpO2 can indicate instability —
a patient whose oxygen saturation is swinging between 91% and 97% repeatedly
is more concerning than one sitting steadily at 94%. The model picking these
up is a good sign.

---

### FiO2 (rank 10) — should be higher, explains itself
FiO2 had the strongest correlation with ventilation in our EDA (0.11) and
covers 96% of ventilated patients. It should arguably be the top feature
clinically — rising oxygen requirement is one of the clearest warning signs
before intubation.

Its low rank here is entirely explained by **58.4% missingness**. In over half
of all patient-hours, FiO2 is NaN because it is primarily recorded once
ventilator support begins. The model cannot learn from a value that is absent
for the majority of rows.

This is a data coverage problem, not a modelling problem. With better FiO2
coverage (or using the ICD-Balancer synthetic data approach to fill gaps),
FiO2 would likely rise to rank 1 or 2 where it belongs clinically.

---

### Overall interpretation
The feature importance pattern is **partially clinically valid** — deteriorating
respiratory and cardiac trends appear near the top, which is what a clinician
would look for. However, the dominance of static demographic features (age,
weight, unit type) over real-time signals suggests the model is partly profiling
patients rather than detecting hour-by-hour deterioration.

The main action items arising from this analysis are:
1. Test removing age and weight to see if real-time signals improve in rank
2. Investigate FiO2 coverage to unlock its full predictive potential
3. Compare these importances against the XGBoost model to see if the pattern
   is consistent across models or specific to LightGBM

---
# Part 6 — Hyperparameter Tuning

## Why tune hyperparameters?

The LightGBM model in Part 5 used a **manually chosen set of parameters** — reasonable defaults
that work well in many settings, but not necessarily optimal for our specific data. With extreme
class imbalance (450:1) and very few positive examples (~250 in training), small changes in
parameters can have a large effect on AUPRC.

### What the baseline results showed us

| Metric | Baseline value | What it means |
|---|---|---|
| AUPRC | **0.0037** | 2.1× above random — model has *some* signal but performance is poor |
| Trees used | **100** | Early stopping fired almost immediately — the model is not learning deep patterns |
| Optimal threshold recall | **0.23** | Even with the best threshold, we catch fewer than 1 in 4 danger hours |

The fact that early stopping triggered at 100 trees (out of 1000) tells us the model is **not
improving** — it finds a local minimum and then plateaus. This can happen when:
- The learning rate is too high (jumps over good solutions)
- The tree complexity is too restrictive (`max_depth`, `num_leaves`)
- Regularisation (`reg_alpha`, `reg_lambda`) is too aggressive

Hyperparameter tuning is the systematic search for parameter combinations that help the model
learn better from these imbalanced data.

### Strategy: Bayesian optimisation with Optuna

We use **Optuna**, a modern hyperparameter tuning library, which uses Bayesian optimisation
(specifically the Tree Parzen Estimator algorithm) to intelligently explore the parameter space.
It is much more efficient than grid search because it learns which regions of the parameter
space are promising and focuses trials there.

### Evaluation: 3-fold stratified cross-validation

Each Optuna trial trains 3 LightGBM models on 3 different training/validation splits and
reports the **mean AUPRC** on the held-out fold. This:
- Reduces variance in the estimate (one bad split does not dominate)
- Prevents us from accidentally overfitting to the specific test split
- Preserves the class ratio in each fold (stratified)

### Parameters we tune and why

| Parameter | Search range | Why |
|---|---|---|
| `learning_rate` | 0.001 – 0.3 (log scale) | Lower rates often generalise better but need more trees |
| `n_estimators` | 200 – 2000 | Allow more trees when the learning rate is low |
| `num_leaves` | 15 – 127 | Controls tree complexity; too many = overfitting on rare positives |
| `max_depth` | 3 – 10 | Caps depth; shallower trees generalise better for rare events |
| `min_child_samples` | 5 – 100 | **Critical**: minimum samples per leaf — prevents leaves built around a single positive row |
| `subsample` | 0.4 – 1.0 | Row subsampling; adds randomness, reduces overfitting |
| `colsample_bytree` | 0.4 – 1.0 | Feature subsampling per tree |
| `reg_alpha` | 1e-5 – 10.0 (log scale) | L1 regularisation; can be loosened if the baseline was too tight |
| `reg_lambda` | 1e-5 – 10.0 (log scale) | L2 regularisation; same reason |


In [ ]:
# Install Optuna if not already present — uncomment if needed
# !pip install optuna --quiet

import optuna
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score as _auprc

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Objective function ────────────────────────────────────────────────────────
# Each Optuna 'trial' calls this function with a new set of hyperparameters.
# We train on 3 cross-validation folds and return the mean AUPRC.
# Optuna maximises this value across trials.

def objective(trial):
    params = {
        'objective':         'binary',
        'metric':            'average_precision',
        'verbose':           -1,
        'n_jobs':            -1,
        'random_state':      42,
        'scale_pos_weight':  spw,          # keep imbalance weighting from Cell 3
        # ── Tuneable parameters ──
        'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.3,  log=True),
        'n_estimators':      trial.suggest_int  ('n_estimators',  200,  2000, step=100),
        'num_leaves':        trial.suggest_int  ('num_leaves',    15,   127),
        'max_depth':         trial.suggest_int  ('max_depth',     3,    10),
        'min_child_samples': trial.suggest_int  ('min_child_samples', 5, 100),
        'subsample':         trial.suggest_float('subsample',     0.4,  1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha',  1e-5, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-5, 10.0, log=True),
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    fold_auprcs = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        sw_tr = sample_weights[train_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            sample_weight=sw_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(stopping_rounds=30, verbose=False),
                lgb.log_evaluation(period=-1),
            ]
        )
        preds = model.predict_proba(X_val)[:, 1]
        fold_auprcs.append(_auprc(y_val, preds))

    return float(np.mean(fold_auprcs))


# ── Run Optuna study ──────────────────────────────────────────────────────────
# n_trials=50 is a reasonable starting point; increase if time allows.
# timeout=600 adds a hard stop at 10 minutes as a safety valve.

study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, timeout=600)

print(f"\n✅ Hyperparameter tuning complete!")
print(f"   Trials run     : {len(study.trials)}")
print(f"   Best CV AUPRC  : {study.best_value:.4f}")
print(f"\n   Best parameters:")
for k, v in study.best_params.items():
    print(f"     {k:25s} = {v}")


## Cell 9 — Interpreting the Tuning Results

### What the best CV AUPRC tells us

The cross-validation AUPRC reported above is **more reliable** than the single test-set AUPRC
from Cell 4. It is the average across three held-out validation folds, so it reflects how
well the *parameter set* generalises rather than how well a specific model fit a specific split.

Compare:
- **Baseline (manually tuned)** single test AUPRC: 0.0037
- **Optuna best** CV AUPRC: *(shown above)*

### How to read parameter changes

When the best parameters differ significantly from the Cell 3 baseline, here is what each
direction of change implies:

| If Optuna chose… | Interpretation |
|---|---|
| **Lower `learning_rate`** | Model needs more gradual steps — the baseline was "overshooting" |
| **More `n_estimators`** | Lower LR requires more trees to converge — makes sense together |
| **More `num_leaves`**   | Data has more complex structure than the baseline allowed |
| **Fewer `min_child_samples`** | Some leaf granularity is useful — baseline was too conservative |
| **Lower `reg_alpha/lambda`** | Baseline regularisation was too tight, suppressing useful signals |


In [ ]:
# ── Train final model with best hyperparameters ───────────────────────────────
# We retrain on the full training set (not just CV folds) so the model uses
# all available data before evaluating on the held-out test set.

best_params_full = {
    'objective':        'binary',
    'metric':           'average_precision',
    'verbose':          -1,
    'n_jobs':           -1,
    'random_state':     42,
    'scale_pos_weight': spw,
    **study.best_params,
}

best_model = lgb.LGBMClassifier(**best_params_full)
best_model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100),
    ]
)

# ── Metrics ───────────────────────────────────────────────────────────────────
y_proba_best = best_model.predict_proba(X_test)[:, 1]
auprc_best   = average_precision_score(y_test, y_proba_best)
auroc_best   = roc_auc_score(y_test, y_proba_best)

# Optimal threshold for the best model
prec_b, rec_b, thresh_b = precision_recall_curve(y_test, y_proba_best)
f1_b        = 2 * prec_b[:-1] * rec_b[:-1] / (prec_b[:-1] + rec_b[:-1] + 1e-8)
best_idx_b  = np.argmax(f1_b)
best_thr_b  = thresh_b[best_idx_b]
y_pred_best = (y_proba_best >= best_thr_b).astype(int)

# ── Side-by-side comparison ───────────────────────────────────────────────────
print("=" * 55)
print("  BASELINE MODEL vs HYPERPARAMETER-TUNED MODEL")
print("=" * 55)
print(f"  {'Metric':<30} {'Baseline':>10} {'Tuned':>10}")
print("-" * 55)
print(f"  {'AUPRC (primary)':<30} {auprc:>10.4f} {auprc_best:>10.4f}")
print(f"  {'AUROC':<30} {auroc:>10.4f} {auroc_best:>10.4f}")
print(f"  {'Improvement over random (AUPRC)':<30} {auprc/y_test.mean():>9.1f}x {auprc_best/y_test.mean():>9.1f}x")
print(f"  {'Trees used':<30} {lgbm_model.best_iteration_:>10} {best_model.best_iteration_:>10}")
print(f"  {'Optimal threshold':<30} {best_threshold:>10.4f} {best_thr_b:>10.4f}")
print()

# Classification report at optimal threshold
print(f"Tuned model — classification report at threshold {best_thr_b:.4f}:")
print(classification_report(y_test, y_pred_best,
      target_names=['safe (0)', 'danger (1)']))

# ── Plot: PR and ROC for both models ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Baseline vs Tuned — Evaluation Curves', fontweight='bold')

PrecisionRecallDisplay.from_predictions(
    y_test, y_pred_proba, ax=axes[0], name='Baseline')
PrecisionRecallDisplay.from_predictions(
    y_test, y_proba_best, ax=axes[0], name='Tuned')
axes[0].axhline(y=y_test.mean(), color='red', linestyle='--',
                label=f'Random baseline ({y_test.mean():.4f})')
axes[0].set_title('Precision-Recall Curve')
axes[0].legend()

RocCurveDisplay.from_predictions(
    y_test, y_pred_proba, ax=axes[1], name='Baseline')
RocCurveDisplay.from_predictions(
    y_test, y_proba_best, ax=axes[1], name='Tuned')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()


---
# Part 7 — When Results Still Aren't Good Enough

## What to do when AUPRC remains low after hyperparameter tuning

If the tuned model still has AUPRC below 0.05 (our "reasonable" threshold for this data),
we have several additional levers to pull. Below we explain each strategy, when to use it,
and what to watch out for.

---

## Strategy 1 — SMOTE / Oversampling the minority class

### What it is
SMOTE (Synthetic Minority Oversampling TEchnique) creates **synthetic positive examples** by
interpolating between existing positive rows in feature space. Instead of the model seeing
250 positive rows, it might see 2,500 or 5,000 synthetic ones.

### Why it might help here
Our training set has only 250 positive rows. At this scale, the model struggles to learn
reliable patterns — a few noisy positive examples dominate the signal. SMOTE gives the model
more examples to generalise from.

### Critical warning for time-series data
SMOTE must **only be applied to the training set, never the test set**. Applying it to the test
set would simulate patients that do not exist and produce misleadingly high metrics.

Additionally, if you are using time-based splits, apply SMOTE **inside the training fold only**
to avoid information leakage from future timestamps.

### Alternatives to SMOTE
- **Random oversampling**: Simply duplicate existing positive rows (simpler, sometimes equally effective)
- **ADASYN**: Like SMOTE but focuses synthetic examples on harder-to-classify positives
- **Class weights only**: What we already use with `scale_pos_weight` — sometimes sufficient

---

## Strategy 2 — More aggressive feature engineering

The current features use 3-hour rolling windows. Consider:

| New feature type | Clinical rationale |
|---|---|
| **6-hour and 12-hour rolling means/std** | Slower deterioration trends visible over longer windows |
| **Trend slope** (linear regression over last N readings) | Captures *direction* of change, not just magnitude |
| **Time since last abnormal value** | A patient 4 hours out from high FiO2 vs 20 hours is very different |
| **Cross-feature interactions** (e.g. FiO2 × respiration_rate) | High FiO2 *combined with* high respiratory rate is more alarming than either alone |
| **Missingness as a feature** | A suddenly missing FiO2 after it was being recorded may itself signal deterioration |

---

## Strategy 3 — Revisit the problem formulation

If the model cannot learn from the current formulation, the issue may not be algorithmic.
Consider:

- **Expanding the positive window**: Currently, a "danger hour" is an hour *immediately* before
  ventilation. Labelling hours within 3, 6, or 12 hours of ventilation as positive gives the
  model a larger positive signal to learn from, and is arguably more clinically useful
  (alerting 6 hours in advance is as valuable as 1 hour).
- **Relaxing the exclusion criteria**: Review which hours are excluded and whether the
  exclusions are too aggressive, removing informative borderline cases.

---

## Strategy 4 — Try a different model class

LightGBM is a strong baseline, but other model families may handle this problem differently:

| Model | When to try it |
|---|---|
| **XGBoost** | Direct comparison — may perform similarly or better with different gradient regularisation |
| **CatBoost** | Handles high cardinality categoricals (unit type, patient ID encoding) natively |
| **Random Forest** | More resistant to overfitting on rare events due to averaging |
| **Logistic Regression** (with polynomial features) | Useful as a sanity-check baseline; interpretable |
| **Neural networks (LSTM/GRU)** | Sequence models that could capture temporal dependencies across an entire ICU stay |

---

## Strategy 5 — Ensemble the baseline and tuned models

If neither the baseline nor the tuned model is clearly better, their predictions can be
**averaged** (probability ensemble). This often reduces variance and improves AUPRC even when
individual models are weak:

```python
y_proba_ensemble = (y_pred_proba + y_proba_best) / 2
auprc_ensemble   = average_precision_score(y_test, y_proba_ensemble)
```

---

## Decision flowchart: what to try next

```
AUPRC after tuning
│
├── < 0.05 (barely above random)
│   ├── Is FiO2 coverage still low?
│   │   └── Yes → Prioritise improving FiO2 coverage / imputation strategy
│   ├── Are static features (age, weight) dominating?
│   │   └── Yes → Drop them and retrain; add more real-time features
│   └── Still stuck → Try expanding positive window (6h / 12h)
│
├── 0.05 – 0.15 (reasonable)
│   ├── Apply SMOTE and compare
│   ├── Try XGBoost / CatBoost for comparison
│   └── Engineer trend / slope features
│
└── > 0.15 (strong)
    └── Focus on threshold selection and clinical validation
```


In [ ]:
# ── Strategy 1: SMOTE oversampling ───────────────────────────────────────────
# Install imbalanced-learn if needed: !pip install imbalanced-learn --quiet

from imblearn.over_sampling import SMOTE

# ── Apply SMOTE to the TRAINING SET ONLY ──────────────────────────────────────
# SMOTE interpolates between existing positive rows to create synthetic ones.
# We target a 10:1 ratio (negative:positive) rather than full balance,
# because fully balancing is usually too aggressive for extreme imbalance.
# sampling_strategy=0.1 means: make positives = 10% of negatives.

desired_ratio = 0.1  # positives become ~10% of negatives
smote = SMOTE(sampling_strategy=desired_ratio, random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {(y_train == 0).sum()} neg / {(y_train == 1).sum()} pos")
print(f"After  SMOTE: {(y_train_sm == 0).sum()} neg / {(y_train_sm == 1).sum()} pos")
print(f"New ratio   : {(y_train_sm == 1).sum() / (y_train_sm == 0).sum():.3f}")

# ── Recompute sample weights for the SMOTE set ────────────────────────────────
# After SMOTE, the class ratio has changed so we recompute effective-sample weights.
class_weights_sm  = effective_sample_weights(y_train_sm)
sample_weights_sm = np.array([class_weights_sm[y] for y in y_train_sm])

# ── Train model on SMOTE data (using best hyperparameters from Optuna) ─────────
# NOTE: when using SMOTE we typically reduce or remove scale_pos_weight because
# the oversampling already handles the imbalance.
smote_params = {k: v for k, v in best_params_full.items()}
smote_params['scale_pos_weight'] = 1  # oversampling replaces this

smote_model = lgb.LGBMClassifier(**smote_params)
smote_model.fit(
    X_train_sm, y_train_sm,
    sample_weight=sample_weights_sm,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100),
    ]
)

# ── Evaluate ──────────────────────────────────────────────────────────────────
y_proba_smote = smote_model.predict_proba(X_test)[:, 1]
auprc_smote   = average_precision_score(y_test, y_proba_smote)
auroc_smote   = roc_auc_score(y_test, y_proba_smote)

print(f"\n{'='*55}")
print("  THREE-WAY COMPARISON")
print(f"{'='*55}")
print(f"  {'Model':<25} {'AUPRC':>8} {'AUROC':>8} {'vs random':>10}")
print("-" * 55)
print(f"  {'Baseline':<25} {auprc:>8.4f} {auroc:>8.4f} {auprc/y_test.mean():>9.1f}x")
print(f"  {'Tuned (Optuna)':<25} {auprc_best:>8.4f} {auroc_best:>8.4f} {auprc_best/y_test.mean():>9.1f}x")
print(f"  {'Tuned + SMOTE':<25} {auprc_smote:>8.4f} {auroc_smote:>8.4f} {auprc_smote/y_test.mean():>9.1f}x")

# ── Threshold tuning for SMOTE model ─────────────────────────────────────────
prec_s, rec_s, thresh_s = precision_recall_curve(y_test, y_proba_smote)
f1_s       = 2 * prec_s[:-1] * rec_s[:-1] / (prec_s[:-1] + rec_s[:-1] + 1e-8)
best_s_idx = np.argmax(f1_s)
best_thr_s = thresh_s[best_s_idx]
y_pred_s   = (y_proba_smote >= best_thr_s).astype(int)

print(f"\nSMOTE model — optimal threshold: {best_thr_s:.4f}")
print(f"  Precision: {prec_s[best_s_idx]:.4f}   Recall: {rec_s[best_s_idx]:.4f}   F1: {f1_s[best_s_idx]:.4f}")
print()
print(classification_report(y_test, y_pred_s, target_names=['safe (0)', 'danger (1)']))


In [ ]:
# ── Strategy 5: Probability ensemble ────────────────────────────────────────
# Average the predicted probabilities from all three models.
# This reduces variance: when one model is uncertain, the others may be more
# confident, and the average lands closer to the truth.

y_proba_ensemble = (y_pred_proba + y_proba_best + y_proba_smote) / 3
auprc_ensemble   = average_precision_score(y_test, y_proba_ensemble)
auroc_ensemble   = roc_auc_score(y_test, y_proba_ensemble)

# Optimal threshold for ensemble
prec_e, rec_e, thresh_e = precision_recall_curve(y_test, y_proba_ensemble)
f1_e       = 2 * prec_e[:-1] * rec_e[:-1] / (prec_e[:-1] + rec_e[:-1] + 1e-8)
best_e_idx = np.argmax(f1_e)
best_thr_e = thresh_e[best_e_idx]
y_pred_e   = (y_proba_ensemble >= best_thr_e).astype(int)

print(f"\n{'='*55}")
print("  FINAL FOUR-WAY COMPARISON (including ensemble)")
print(f"{'='*55}")
print(f"  {'Model':<28} {'AUPRC':>8} {'AUROC':>8} {'vs random':>10}")
print("-" * 58)
print(f"  {'Baseline':<28} {auprc:>8.4f} {auroc:>8.4f} {auprc/y_test.mean():>9.1f}x")
print(f"  {'Tuned (Optuna)':<28} {auprc_best:>8.4f} {auroc_best:>8.4f} {auprc_best/y_test.mean():>9.1f}x")
print(f"  {'Tuned + SMOTE':<28} {auprc_smote:>8.4f} {auroc_smote:>8.4f} {auprc_smote/y_test.mean():>9.1f}x")
print(f"  {'Ensemble (avg)':<28} {auprc_ensemble:>8.4f} {auroc_ensemble:>8.4f} {auprc_ensemble/y_test.mean():>9.1f}x")

print(f"\nEnsemble optimal threshold: {best_thr_e:.4f}")
print(classification_report(y_test, y_pred_e, target_names=['safe (0)', 'danger (1)']))

# ── PR curves for all four models ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
for proba, name in [
    (y_pred_proba,     'Baseline'),
    (y_proba_best,     'Tuned (Optuna)'),
    (y_proba_smote,    'Tuned + SMOTE'),
    (y_proba_ensemble, 'Ensemble'),
]:
    PrecisionRecallDisplay.from_predictions(y_test, proba, ax=ax, name=name)

ax.axhline(y=y_test.mean(), color='red', linestyle='--',
           label=f'Random baseline ({y_test.mean():.4f})')
ax.set_title('Precision-Recall: All Models Compared', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


---
# Part 8 — Summary and Next Steps

## What we did in this notebook

| Step | What | Why |
|---|---|---|
| Part 5 | Baseline LightGBM with manual parameters | Establish a reference point |
| Part 5 | Effective-sample class weighting | Prevent model ignoring rare positives |
| Part 5 | Early stopping | Prevent overfitting to the small positive class |
| Part 5 | AUPRC evaluation | Correct metric for extreme class imbalance |
| Part 5 | Threshold tuning | Default 0.5 threshold is wrong for imbalanced data |
| Part 6 | Optuna hyperparameter tuning (50 trials, 3-fold CV) | Systematic search beats manual guessing |
| Part 7 | SMOTE oversampling on training set only | Give model more positive examples to generalise from |
| Part 7 | Probability ensemble | Reduce variance by averaging model disagreements |

## Key lessons

1. **AUPRC is the only honest metric here.** Accuracy, AUROC, and F1 can all be misleadingly high with 450:1 imbalance.

2. **Threshold tuning is not optional.** A model that predicts `y=0` for everything achieves
   99.78% accuracy. The decision threshold must be chosen to reflect the clinical cost
   trade-off between false alarms and missed events.

3. **Early stopping at 100 trees is a warning sign.** It means the model is not learning.
   Hyperparameter tuning (especially lowering the learning rate) typically resolves this by
   allowing the model to take smaller, more careful steps.

4. **Static features dominating is a problem.** When `age`, `admissionweight`, or `unittype`
   outrank real-time vital signs, the model is identifying *which patients* are risky rather
   than *when* they are deteriorating. Consider dropping them or investigating patient-level leakage.

5. **FiO2 coverage is a data quality bottleneck.** At 58% missingness, one of the strongest
   clinical predictors cannot be fully learned. Improving FiO2 coverage (e.g. via ICD-Balancer
   imputation or different data sources) could substantially improve all models.

## Recommended next steps (by priority)

1. **Remove static demographic features** (age, weight) and retrain — if AUPRC barely changes,
   they were acting as shortcuts around the real-time signal
2. **Improve FiO2 coverage** — talk to data engineering about alternative sources
3. **Expand the positive window** from 1 hour to 6–12 hours before ventilation
4. **Engineer trend/slope features** over 6h and 12h windows
5. **Run this comparison on XGBoost** and compare feature importances across both models
